# `c05_c` — Completions by Program, Level, and Demographic Group

**Component curation notebook.** Fetches the raw IPEDS distribution files, verifies the
reference period against official documentation, locks the schema, reshapes to the
declared grain, validates, and writes one curated table with a metadata sidecar.

| Property | Value |
|---|---|
| Native tables | `C2023_A`, `C2023_B`, `C2023_C`, `DRVC2023` |
| Reference period | Awards conferred July 1, 2022 through June 30, 2023 |
| Curated grain | `UNITID` x `CIPCODE` x `AWLEVEL` x `MAJORNUM` |
| Output | `data/curated/c05_c.parquet` |

C2023_A counts awards, not people, and stores totals beside their detail: each UNITID x AWLEVEL x MAJORNUM has a CIPCODE == '99' row equal to the sum of its programme rows. Summing CTOTALT over every row therefore counts each award twice; the rolls_up rule below checks the identity. Institution award totals come from the CIPCODE '99' rows with MAJORNUM == 1, since second majors are not additional awards. CIPCODE is read as text: parsed as a number, 13.0100 becomes 13.01 and 01.0000 becomes 1.0. Unique completers are a separate file, C2023_C.

> **Pitfall.** The 2020 CIP revision breaks program-level time series. Any completions trend spanning 2019 to 2020 must be built on a CIP crosswalk, and program-level changes across that boundary are otherwise taxonomy artefacts rather than real shifts in what institutions award.

## 1. Environment

One import surface, so a parsing quirk is fixed once rather than twelve times.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import ipeds_utils as iu

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

SLUG = "c05_c"
TABLES = ['C2023_A', 'C2023_B', 'C2023_C', 'DRVC2023']
GRAIN = ['UNITID', 'CIPCODE', 'AWLEVEL', 'MAJORNUM']
REFERENCE_PERIOD = 'Awards conferred July 1, 2022 through June 30, 2023'

print("ipeds_utils", iu.__version__, "| pandas", pd.__version__)

ipeds_utils 1.1.0 | pandas 3.0.5


## 2. Retrieve

Downloads are cached, so re-running this notebook is offline and cheap. Every retrieval returns a provenance record carrying a SHA-256 digest, which is what makes a result reproducible rather than merely repeatable.

In [2]:
RAW_DIR = "../data/raw"   # relative to notebooks/, so all twelve share one cache

provenance = [iu.fetch(t, raw_dir=RAW_DIR) for t in TABLES]
pd.DataFrame(provenance)[["table", "data_bytes", "data_sha256", "retrieved_utc"]]

,table,data_bytes,data_sha256,retrieved_utc
0,C2023_A,9294160,651d95b6405bb86c6c14884ed54225a27492199d21d8ac...,2026-09-24T17:19:19+00:00
1,C2023_B,500322,a901a43fa4ae1dc7e1fb061eaef9244ce34c157e1dac7c...,2026-09-24T17:19:19+00:00
2,C2023_C,748858,6e5d545569413b691582e71d2f2f76cc43aec1b28ecb4e...,2026-09-24T17:19:19+00:00
3,DRVC2023,82029,1d90bbe5010c2e3cc5c295b5cb1224bd0f44606dc9696a...,2026-09-24T17:19:19+00:00


## 3. Verify the reference period

**Do not skip this cell.** The filename year is not the reference period, and the offsets are not uniform across components. This assertion fails loudly rather than letting a misaligned period corrupt every downstream year comparison, where it would be invisible in the data itself.

In [3]:
intro = iu.assert_reference_period(
    provenance[0]["dict_path"],
    expect=r'(July 1, 2022|2022-23)',
    table=TABLES[0],
)
print(intro[:600])

File Documentation for the Completions Data File, 2022-23
(Final/revised release)
Filename C2023_A
Provisional release: August 2024
Filename C2023_A_RV
Final/revised release: September 2025
Overview This file contains the number of awards by type of program, level of award (certificate or degree), first or second major and by race/ethnicity and gender. Data covers all awards granted between July 1, 2022 and June 30, 2023.  Type of program is categorized according to the 2020 Classification of Instructional Programs (CIP), a detailed coding system for postsecondary instructional programs.  The 


## 4. Inspect the dictionary

Variable labels come from the published dictionary, never from memory. This is also where value sets are read, so categorical decoding is driven by the official codebook and a taxonomy revision surfaces as unmatched codes instead of a plausible-looking wrong label.

In [4]:
variables = iu.read_dict(provenance[0]["dict_path"])
valuesets = iu.read_valuesets(provenance[0]["dict_path"])

print(f"{len(variables)} variables documented, {len(valuesets)} value-set rows")
variables[["varname", "vartitle"]].head(20)

34 variables documented, 1612 value-set rows


,varname,vartitle
0,UNITID,Unique identification number of the institution
1,CIPCODE,CIP Code - 2020 Classification
2,MAJORNUM,First or Second Major
3,AWLEVEL,Award Level code
4,CTOTALT,Grand total
5,CTOTALM,Grand total men
6,CTOTALW,Grand total women
7,CAIANT,American Indian or Alaska Native total
8,CAIANM,American Indian or Alaska Native men
9,CAIANW,American Indian or Alaska Native women


## 5. Load and lock the schema

The first run records the column signature; later runs fail if it drifts.

In [5]:
KEEP = ['UNITID', 'CIPCODE', 'MAJORNUM', 'AWLEVEL', 'CTOTALT', 'CTOTALM', 'CTOTALW']

raw = iu.read_csv(provenance[0]["data_path"])
print("raw shape", raw.shape)

lock = iu.lock_schema(raw, TABLES[0], schema_dir="../schemas", strict=False)
print("schema:", lock["status"], "| added", lock["added"][:5], "| removed", lock["removed"][:5])

available = [c for c in KEEP if c in raw.columns]
missing = [c for c in KEEP if c not in raw.columns]
if missing:
    print("NOT PRESENT in this cycle (verify against the varlist above):", missing)

frame = raw[available].copy()
frame.head()

raw shape (303460, 64)
schema: unchanged | added [] | removed []


,UNITID,CIPCODE,MAJORNUM,AWLEVEL,CTOTALT,CTOTALM,CTOTALW
0,100654,01.0999,1,5,18,1,17
1,100654,01.1001,1,5,8,2,6
2,100654,01.1001,1,7,6,2,4
3,100654,01.1001,1,17,2,2,0
4,100654,01.9999,1,5,2,0,2


## 6. Mask reserved missing codes

IPEDS encodes missingness as negative integers. A mean computed without masking them is badly wrong and looks entirely plausible, which is what makes this the most costly single omission in IPEDS analysis.

In [6]:
RESERVED = [-1, -2, -3, -9]

numeric_cols = [
    c for c in frame.columns
    if c not in ("UNITID", *GRAIN) and pd.api.types.is_numeric_dtype(frame[c])
]

before = frame[numeric_cols].isna().sum().sum()
for col in numeric_cols:
    frame.loc[frame[col].isin(RESERVED), col] = np.nan
after = frame[numeric_cols].isna().sum().sum()

# Masking turns an integer column into float (1 becomes 1.0). Measures can stay float,
# since NaN is what the models expect, but category codes go back to nullable integers
# so they print, join, and decode as codes rather than as 1.0.
for col in ['AWLEVEL']:
    if col in frame.columns and pd.api.types.is_float_dtype(frame[col]):
        if (frame[col].dropna() % 1 == 0).all():
            frame[col] = frame[col].astype("Int64")

print(f"masked {after - before:,} reserved-code cells across {len(numeric_cols)} numeric columns")

masked 0 reserved-code cells across 3 numeric columns


## 7. Carry the imputation flags

An imputed value and a reported value are not the same evidence. A column where most institutions carry a generated flag should not be modelled as though it were observed, and this is where that judgement becomes possible.

In [7]:
values, flags = iu.split_imputation_flags(raw, numeric_cols)

if flags.shape[1] > 1:
    summary = iu.imputation_summary(flags)
    display(summary.head(15))
    reported = summary[summary.flag == "R"].set_index("column")["share"]
    weak = reported[reported < 0.90]
    if len(weak):
        print("Columns under 90% reported — interpret with care:")
        display(weak)
else:
    print("No X-prefixed imputation flags accompany this file.")

,column,flag,n,share
2,XCTOTALM,R,275872,0.9091
3,XCTOTALM,Z,27588,0.0909
0,XCTOTALT,R,303458,1.0000
1,XCTOTALT,C,2,0.0000
4,XCTOTALW,R,257296,0.8479
5,XCTOTALW,Z,46162,0.1521
6,XCTOTALW,C,2,0.0000


Columns under 90% reported — interpret with care:


column
XCTOTALW    0.8479
Name: share, dtype: float64

## 8. Decode categoricals

Labels from the published value sets, not hand-typed mappings.

In [8]:
CATEGORICALS = ['AWLEVEL']

unresolved = {}
for col in CATEGORICALS:
    if col in frame.columns:
        frame = iu.decode(frame, valuesets, col)
        unmatched = frame.loc[frame[col].notna() & frame[f"{col}_LABEL"].isna(), col].unique()
        if len(unmatched):
            unresolved[col] = sorted(unmatched.tolist())[:10]

# An unmatched code means a taxonomy change or a parsing fault. Either way the
# labels are wrong, so this stops the notebook rather than printing a warning.
assert not unresolved, f"codes absent from the published value set: {unresolved}"

label_cols = [c for c in frame.columns if c.endswith("_LABEL")]
frame[CATEGORICALS + label_cols].drop_duplicates().head(20) if label_cols else frame.head()

,AWLEVEL,AWLEVEL_LABEL
0,5,Bachelor's degree
2,7,Master's degree
3,17,Doctor's degree - research/scholarship
14,8,Post-master's certificate
70,6,Postbaccalaureate certificate
128,2,Certificates of at least 1 but less than 2 years
219,18,Doctor's degree - professional practice
235,21,Certificates of at least 12 weeks but less tha...
304,3,Associate's degree
691,4,Certificates of at least 2 but less than 4 years


## 9. Reshape to the declared grain

Target grain: `UNITID` x `CIPCODE` x `AWLEVEL` x `MAJORNUM`. The grain is asserted, not assumed, because a duplicated key silently inflates every aggregate computed downstream.

In [9]:
curated = frame.copy()

# This component already arrives at its declared grain, so curation is a
# pass-through. Components with a long layout (GRTYPE, EFFYALEV, STAFFCAT,
# OMCHRT) filter or pivot here instead; see c10_f for a full worked reshape.

present_grain = [g for g in GRAIN if g in curated.columns]
duplicated = curated.duplicated(subset=present_grain, keep=False).sum()
print(f"grain {present_grain} -> {len(curated):,} rows, {duplicated} duplicated")
assert duplicated == 0, "Declared grain is not unique; resolve before continuing."

curated.head()

grain ['UNITID', 'CIPCODE', 'AWLEVEL', 'MAJORNUM'] -> 303,460 rows, 0 duplicated


,UNITID,CIPCODE,MAJORNUM,AWLEVEL,CTOTALT,CTOTALM,CTOTALW,AWLEVEL_LABEL
0,100654,01.0999,1,5,18.0,1.0,17.0,Bachelor's degree
1,100654,01.1001,1,5,8.0,2.0,6.0,Bachelor's degree
2,100654,01.1001,1,7,6.0,2.0,4.0,Master's degree
3,100654,01.1001,1,17,2.0,2.0,0.0,Doctor's degree - research/scholarship
4,100654,01.9999,1,5,2.0,0.0,2.0,Bachelor's degree


## 10. Validate

Rules are declarative so the output is a persistable report: which checks ran, which failed, on how many rows, and which institutions were implicated. That report is the artefact you cite when claiming this table is fit for analysis.

In [10]:
RULES = [
    iu.unique_key('UNITID', 'CIPCODE', 'AWLEVEL', 'MAJORNUM'),
    iu.in_range('CTOTALT', 0, None),
    iu.rolls_up('CTOTALT', level='CIPCODE', total_code='99', keys=['UNITID', 'AWLEVEL', 'MAJORNUM']),
]

report = iu.validate(curated, RULES, SLUG)
report.save(f"../reports/validation/{SLUG}.json")
display(report.to_frame()[["name", "status", "n_offending", "share", "note"]])

print("PASSED" if report.ok else "FAILED")
report.raise_if_failed()

,name,status,n_offending,share,note
0,"unique_key(UNITID,CIPCODE,AWLEVEL,MAJORNUM)",pass,0,0.0,Declared grain must be unique
1,"in_range(CTOTALT,0,None)",pass,0,0.0,Value plausibility bound
2,"rolls_up(CTOTALT, CIPCODE==99)",pass,0,0.0,CIPCODE 99 rows must equal the sum of detail r...


PASSED


Report(table='c05_c', rows=303460, results=[{'name': 'unique_key(UNITID,CIPCODE,AWLEVEL,MAJORNUM)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Declared grain must be unique', 'status': 'pass'}, {'name': 'in_range(CTOTALT,0,None)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Value plausibility bound', 'status': 'pass'}, {'name': 'rolls_up(CTOTALT, CIPCODE==99)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': "CIPCODE 99 rows must equal the sum of detail rows within ['UNITID', 'AWLEVEL', 'MAJORNUM']", 'status': 'pass'}], generated_utc='2026-09-24T17:19:22+00:00')

## 11. Write the curated table

The sidecar carries the reference period and grain with the data. This is the defence against assembling a panel by filename year when the underlying periods are offset differently per component.

In [11]:
path = iu.write_curated(
    curated,
    SLUG,
    root="../data/curated",
    reference_period=REFERENCE_PERIOD,
    grain=GRAIN,
    provenance=provenance,
    notes='The 2020 CIP revision breaks program-level time series. Any completions trend spanning 2019 to 2020 must be built on a CIP crosswalk, and program-level changes across that boundary are otherwise taxonomy artefacts rather than real shifts in what institutions award.',
)

iu.write_provenance(provenance, f"../docs/provenance/{SLUG}.json")
print("wrote", path, f"({len(curated):,} rows x {curated.shape[1]} columns)")

wrote ../data/curated/c05_c.parquet (303,460 rows x 8 columns)


## 12. Exercises

1. Re-run this notebook against the prior collection cycle by changing `TABLES`. The schema lock and the period assertion will both object; resolve each objection and record what changed between cycles.
2. Identify the three columns with the lowest reported-flag share, and argue whether each belongs in a predictive model at all.
3. Construct one derived cross-tabulation from this table, then apply `iu.suppress` and `iu.k_anonymity` to it. Report the smallest equivalence class before and after coarsening, and state the k you would require before publishing.
4. The 2020 CIP revision breaks program-level time series. Write a validation rule that would catch this error if a colleague made it, and add it to `RULES` above.